In [1]:
# === Setup en verbinding ===
debugging_mode = True

from pyspark.sql.types import StructType, StructField, StringType, IntegerType
import ConnectionConfig as cc


In [2]:
#  === Setup en verbinding ===
cc.setupEnvironment()  # Config setup
cc.setupEnvironment()
print(cc.config.sections())

spark = cc.startLocalCluster("DIM_RAIN")
spark.getActiveSession()
cc.config.read('config.ini')
cc.set_connectionProfile("catchem")


Environment variables are set...
Environment variables are set...
['default', 'tutorial_op', 'catchem', 'kafka']


In [3]:
# Transform
rain_schema = StructType([
    StructField("RainSurKey", IntegerType(), False),
    StructField("RainCode", StringType(), False),
    StructField("RainDescription", StringType(), False)
])


In [4]:
# TRANSFORM
rain_data = [
    (1, "RAIN", "Weer met regen (codes 200-699)"),
    (2, "NORAIN", "Weer zonder regen"),
    (3, "UNKNOWN", "Regen situatie onbekend")
]

rain_dim_df = spark.createDataFrame(rain_data, schema=rain_schema)

if debugging_mode:
    print("Rain dimensie preview:")
    rain_dim_df.show(truncate=False)


Rain dimensie preview:
+----------+--------+------------------------------+
|RainSurKey|RainCode|RainDescription               |
+----------+--------+------------------------------+
|1         |RAIN    |Weer met regen (codes 200-699)|
|2         |NORAIN  |Weer zonder regen             |
|3         |UNKNOWN |Regen situatie onbekend       |
+----------+--------+------------------------------+



In [5]:
# LOAD
rain_dim_df.coalesce(1).write.format("delta").mode("overwrite").save("delta/RAIN_DIM")

if debugging_mode:
    print("Rain dimensie succesvol opgeslagen naar delta/RAIN_DIM")


Rain dimensie succesvol opgeslagen naar delta/RAIN_DIM


In [6]:
spark.stop()